In [1]:
import numpy as np
import pymc as pm
import pytensor
import pytensor.tensor as pt
import matplotlib.pyplot as plt

In [2]:
class Detector():
    def __init__(self, R: float, N: int, r: float, a: float):
        """
        Parameters
        ----------
        R : float
            Радиус детектора в метрах.
        N : int
            Количество фотоумножителей.
        r : float
            Радиус фотокатода в метрах.
        a : float
            Эффективность фотокатода.
        """
        self.R = R
        self.N = N
        self.r = r
        self.a = a
        self.rng = np.random.default_rng()

    def coord_cart(self, theta: float, phi: float) -> np.ndarray:
        return np.array([self.R * np.sin(theta)*np.cos(phi), self.R * np.sin(theta)*np.sin(phi), self.R*np.cos(theta)])

    def fill_detector(self):
        self.photo_multipliers = np.zeros((self.N, 3))
        S = 4 * self.R**2 * np.pi 
        delta_S = S / self.N 
        lin_size = delta_S**0.5 
        N_theta = np.pi * self.R / lin_size 
        delta_theta = np.pi / N_theta

        theta_floor = [int(np.floor(N_theta)), np.pi / np.floor(N_theta), delta_theta - np.pi / np.floor(N_theta)]
        theta_ceil = [int(np.ceil(N_theta)), np.pi / np.ceil(N_theta), delta_theta - np.pi / np.ceil(N_theta)]

        best_theta = min(theta_floor, theta_ceil, key=lambda x: abs(x[2]))
        N_theta, delta_theta = best_theta[0], best_theta[1]
        self.delta_theta = delta_theta

        mid_thetas = np.array([delta_theta / 2 + delta_theta*i for i in range(0, N_theta)])
        self.thetas = mid_thetas
        n_phi = np.array([(np.pi * self.N)**0.5*np.sin(theta) for theta in mid_thetas])

        A = self.N / sum([np.sin(theta) for theta in mid_thetas])
        n_phi = np.rint(np.array([A * np.sin(theta) for theta in mid_thetas])) #-> (3, 7, 10, 10, 7, 3)
        #n_phi = np.array([2, 7, 11, 11, 7, 2])  
        self.n_phis = n_phi.astype(int) 
        self.detectors_before = np.zeros(N_theta).astype(int)
        n = 0
        for i in range(N_theta):
            self.detectors_before[i] = n
            n += n_phi[i]

        phi_phases = np.zeros(N_theta)
        for i in range(N_theta):
            if i == 0:
                continue
            phi_phases[i] = np.pi / np.lcm(int(n_phi[i-1]), int(n_phi[i]))
        self.phis = np.zeros(N_theta)
        n = 0
        phase = 0
        for i in range(N_theta):
            if i < N_theta/2:
                phase += phi_phases[i]
            else:
                phase -= phi_phases[i]
            self.phis[i] = phase
            for j in range(int(n_phi[i])):
                self.photo_multipliers[n] = self.coord_cart(mid_thetas[i], 2*np.pi/n_phi[i] * j + phase)
                n += 1

In [3]:
class Source:
    def __init__(self, pos : np.ndarray, E : float, decay_length : float):
        """
        Parameters
        ----------
        pos : np.array
            координаты источника
        """
        self.Y = 10000 
        self.E = E
        self.pos = pos
        self.decay_lenght = decay_length

In [4]:
detector = Detector(R=2, N=40, r=0.1, a=0.5)
detector.fill_detector()
source = Source(np.zeros(3), 1, 8)

E_min, E_max = 0.6, 1.4
N_max = int(np.ceil(E_max * source.Y))

In [5]:
with pm.Model() as model:
    E = pm.Uniform("E", lower=E_min, upper=E_max)

    u_pos = pm.Uniform("u_pos", 0, 1)
    phi_pos = pm.Uniform("phi_pos", 0, 2*np.pi)
    cos_theta_pos = pm.Uniform("cos_theta_pos", -1, 1)

    rho = detector.R * u_pos**(1/3)
    sin_theta_pos = pt.sqrt(1 - cos_theta_pos**2)
    pos = pm.Deterministic("pos", rho * pt.stack([sin_theta_pos * pt.cos(phi_pos), sin_theta_pos * pt.sin(phi_pos), cos_theta_pos]))

    n_photons = pt.floor(E * source.Y).astype("int64")
    active = pt.arange(N_max) < n_photons

    phi = pm.Uniform("phi", lower=0, upper=2*np.pi, shape=N_max)
    cos_theta = pm.Uniform("cos_theta", lower=-1, upper=1, shape=N_max)
    sin_theta = pt.sqrt(1 - cos_theta**2)
    photons_directions = pt.stack([sin_theta * pt.cos(phi), sin_theta * pt.sin(phi), cos_theta], axis=1)

    nr = photons_directions @ pos
    r2 = pt.dot(pos, pos)
    t = -nr + pt.sqrt(nr**2 + detector.R**2 - r2)

    if source.decay_lenght == 0:
        survived = pt.ones_like(t, dtype="int64")
    else:
        exps = pt.exp(-t / source.decay_lenght)
        u_abs = pm.Uniform("u_abs", 0, 1, shape=N_max)
        survived = pt.cast(u_abs < exps, "int64")

    hits = pos + photons_directions * t[:, None]
    hits_thetas = pt.arccos(pt.clip(hits[:, 2] / detector.R, -1, 1))
    hits_phis = pt.arctan2(hits[:, 1], hits[:, 0])

    theta_indices = pt.clip(pt.floor(hits_thetas / detector.delta_theta).astype("int64"), 0, len(detector.n_phis) - 1)
    n_phis = pt.as_tensor_variable(detector.n_phis)
    phis = pt.as_tensor_variable(detector.phis)
    detectors_before = pt.as_tensor_variable(detector.detectors_before)
    photo_multipliers = pt.as_tensor_variable(detector.photo_multipliers)

    phi_indices = (pt.round(n_phis[theta_indices] / (2*np.pi) * (hits_phis - phis[theta_indices])).astype("int64") % n_phis[theta_indices])
    photo_multiplier_ids = detectors_before[theta_indices] + phi_indices
    photo_multiplier_coords = photo_multipliers[photo_multiplier_ids]

    distances2 = pt.sum((hits - photo_multiplier_coords)**2, axis=1)
    registered = distances2 <= detector.r**2
    valid = pt.cast(registered & active, "int64") * survived

    n_per_pmt = pt.bincount(photo_multiplier_ids, weights=valid, minlength=detector.N)

    charges_data = pm.Data("charges_data", np.zeros(detector.N, dtype="int64"))
    charges = pm.Poisson("charges", mu=detector.a * n_per_pmt + 1e-6, observed=charges_data)

In [6]:
with model:
    samples = pm.sample_prior_predictive(draws=1, var_names=["E", "pos", "charges"], random_seed=42, return_inferencedata=False)

E_true = samples["E"][0]
pos_true = samples["pos"][0]
charges_observed = samples["charges"][0]

print("Энергия:", E_true)
print("Положение:", pos_true)
print("Заряды:", charges_observed)
print("Суммарный заряд:", charges_observed.sum())

Sampling: [E, charges, cos_theta, cos_theta_pos, phi, phi_pos, u_abs, u_pos]


Энергия: 1.1089484900050055
Положение: [-0.44858088  0.09292374  0.69103895]
Заряды: [ 0  7  4  0  6 11  6 10  1  3  0  0  7  2  7  1  1  6  0  2  2  1  0  5
  0  0  5  0  6  0  2  1  0  4  0  3  2  1  3  1]
Суммарный заряд: 110


In [7]:
with model:
    pm.set_data({"charges_data": charges_observed})

with model:
    trace = pm.sample(draws=10000, tune=500, step=pm.DEMetropolisZ(), var_names=["E", "pos"], return_inferencedata=False)
E_samples = trace.get_values("E")
pos_samples = trace.get_values("pos")

print("Истинная энергия:", E_true)
print("Восстановленная энергия:", np.mean(E_samples))
print("Истинное положение:", pos_true)
print("Восстановленное положение:", np.mean(pos_samples, axis=0))
print("Принятые шаги:", np.mean(trace.get_sampler_stats("accepted")))

/var/folders/c3/lnm_9b8j0lv8k85746bpf5mh0000gn/T/ipykernel_73981/1776886163.py:5: FutureWarning: `return_inferencedata=False` is deprecated and will be removed in a future release. Use the default `return_inferencedata=True` and work with the returned `InferenceData` object.
  trace = pm.sample(draws=10000, tune=500, step=pm.DEMetropolisZ(), var_names=["E", "pos"], return_inferencedata=False)
Multiprocess sampling (4 chains in 4 jobs)
DEMetropolisZ: [E, u_pos, phi_pos, cos_theta_pos, phi, cos_theta, u_abs]


Output()

Sampling 4 chains for 500 tune and 10_000 draw iterations (2_000 + 40_000 draws total) took 19 seconds.
The rhat statistic is larger than 1.01 for some parameters. This indicates problems during sampling. See https://arxiv.org/abs/1903.08008 for details
The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable rhat and ess computation. See https://arxiv.org/abs/1903.08008 for details


Истинная энергия: 1.1089484900050055
Восстановленная энергия: 1.0174787762124988
Истинное положение: [-0.44858088  0.09292374  0.69103895]
Восстановленное положение: [-1.3557229   0.46113484  0.03507176]
Принятые шаги: 0.38465


In [8]:
with model:
    posterior = pm.sample(draws=100, tune=100, var_names=["E", "pos"],).posterior

E_samples = posterior["E"].values.flatten()
pos_samples = posterior["pos"].values.reshape(-1, 3)

print("Истинная энергия:", E_true)
print("Восстановленная энергия:", np.mean(E_samples))
print("Истинное положение:", pos_true)
print("Восстановленное положение:", np.mean(pos_samples, axis=0))

E_chains = posterior["E"].values
pos_chains = posterior["pos"].values

for i in range(4):
    print(f"Цепь {i+1}")
    print("E:", E_chains[i].mean())
    print("pos:", pos_chains[i].mean(axis=0))

Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [E, u_pos, phi_pos, cos_theta_pos, phi, cos_theta, u_abs]


Output()

Sampling 4 chains for 100 tune and 100 draw iterations (400 + 400 draws total) took 160 seconds.
Chain 0 reached the maximum tree depth. Increase `max_treedepth`, increase `target_accept` or reparameterize.
Chain 1 reached the maximum tree depth. Increase `max_treedepth`, increase `target_accept` or reparameterize.
Chain 2 reached the maximum tree depth. Increase `max_treedepth`, increase `target_accept` or reparameterize.
Chain 3 reached the maximum tree depth. Increase `max_treedepth`, increase `target_accept` or reparameterize.
The rhat statistic is larger than 1.01 for some parameters. This indicates problems during sampling. See https://arxiv.org/abs/1903.08008 for details
The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable rhat and ess computation. See https://arxiv.org/abs/1903.08008 for details


Истинная энергия: 1.1089484900050055
Восстановленная энергия: 0.9248375470491464
Истинное положение: [-0.44858088  0.09292374  0.69103895]
Восстановленное положение: [-0.2456082   0.0477618   0.46424198]
Цепь 1
E: 1.1918743829918126
pos: [-0.32675686  0.09338849  0.74334774]
Цепь 2
E: 0.7682917493479011
pos: [-0.85849155 -0.13926627 -0.11686985]
Цепь 3
E: 0.9998221767643521
pos: [0.02275284 0.90264054 0.49469901]
Цепь 4
E: 0.7393618790925202
pos: [ 0.18006277 -0.66571554  0.73579103]
